In [1]:
!pip install torch torchaudio transformers soundfile pandas

In [2]:
import torch
import torch.nn as nn
import torchaudio
import os
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoFeatureExtractor, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, BertTokenizer, Wav2Vec2Processor, Wav2Vec2FeatureExtractor
import numpy as np
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from torch.utils.data import Dataset
import tensorflow_hub as hub
from transformers import TrainerCallback
import pandas as pd

In [3]:
!cp -r '/kaggle/input/splits-updated-223/splits/DAIC_WOZ' '/kaggle/working/'
!mkdir '/kaggle/working/model'

!cp '/kaggle/input/pseudo-labels-for-daic-woz-complete/daic_labels_combined.csv' '/kaggle/working/daic_labels.csv'


In [4]:
!cp '/kaggle/input/bert-pretrained/bert_meld_finetune_model.pth' '/kaggle/working/model/bert_meld_finetune_model.pth'

In [5]:
!cp '/kaggle/input/multimodel/pytorch/default/1/model_checkpoint.pth' '/kaggle/working/model/multimodal_MELD.pth'

In [6]:
from sklearn.metrics import accuracy_score
from tqdm import tqdm

In [7]:
label_df = pd.read_csv('/kaggle/working/daic_labels.csv')

def match_label(audio_file):
    try:
        label_row = label_df[label_df['file'] == audio_file]
        raw_label = label_row['text_predicted'].values[0]
        
        # 处理空值和未知标签
        if pd.isnull(raw_label):
            return "neutral"
        return str(raw_label).strip().lower() if str(raw_label).strip().lower() in [
            "neutral", "joy", "sadness", "anger", "surprise", "fear", "disgust"
        ] else "neutral"
    except Exception as e:
        print(f"Error processing {audio_file}: {str(e)}")
        return "neutral"  # 默认返回中性标签


In [8]:
class DAIC_WOZ_Modal_Dataset(Dataset):
    def __init__(self, audio_dir, text_dir, audio_processor, text_tokenizer):
        self.audio_dir = audio_dir
        self.text_dir = text_dir
        self.audio_processor = audio_processor
        self.text_tokenizer = text_tokenizer

        # 获取原始文件列表
        self.audio_files = [f for f in os.listdir(audio_dir) if f.endswith(".wav")]
        self.text_files = [f.replace('.wav', '.txt') for f in self.audio_files]
        
        # 标签映射
        self.label_map = {
            "neutral": 0,
            "joy": 1,
            "sadness": 2,
            "anger": 3,
            "surprise": 4,
            "fear": 5,
            "disgust": 6
        }

        # 过滤无效样本
        self.valid_indices = []
        for idx in tqdm(range(len(self.audio_files)), desc="过滤无效数据"):
            audio_file = self.audio_files[idx]
            label = match_label(audio_file)
            if label in self.label_map:  # 只保留有效标签
                self.valid_indices.append(idx)
        
        print(f"原始样本数: {len(self.audio_files)}, 有效样本数: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        real_idx = self.valid_indices[idx]
        audio_file = self.audio_files[real_idx]
        audio_path = os.path.join(self.audio_dir, audio_file)
        
        # 加载并处理音频（保持Tensor格式）
        speech_array, sampling_rate = torchaudio.load(audio_path)
        if sampling_rate != 16000:
            resampler = torchaudio.transforms.Resample(sampling_rate, 16000)
            speech_array = resampler(speech_array)
        
        # 确保处理后的音频是Tensor
        audio_inputs = self.audio_processor(
            speech_array.squeeze(), 
            sampling_rate=16000, 
            return_tensors="pt",  # 关键：返回PyTorch Tensor
            padding="max_length",
            max_length=16000 * 3,
            truncation=True
        )
        audio_values = audio_inputs["input_values"].squeeze()
        
        # 使用PyTorch进行归一化（保持Tensor格式）
        audio_mean = torch.mean(audio_values)
        audio_std = torch.std(audio_values)
        normalized_audio = (audio_values - audio_mean) / (audio_std + 1e-8)

        # 处理文本
        text_file = self.text_files[real_idx]
        text_path = os.path.join(self.text_dir, text_file)
        with open(text_path, 'r') as f:
            text_content = f.read().strip()
        
        text_inputs = self.text_tokenizer(
            text_content,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"  # 返回Tensor
        )

        # 获取标签
        label = match_label(audio_file)
        label_id = self.label_map[label]

        return {
            "audio_input_values": normalized_audio,  # 已经是Tensor
            "text_input_ids": text_inputs["input_ids"].squeeze(),
            "text_attention_mask": text_inputs["attention_mask"].squeeze(),
            "labels": torch.tensor(label_id, dtype=torch.long)
        }

In [9]:
train_audio_dir = "/kaggle/input/splits-updated-223/splits/DAIC_WOZ/train/audio"
train_text_dir = "/kaggle/input/splits-updated-223/splits/DAIC_WOZ/train/text"
val_audio_dir = "/kaggle/input/splits-updated-223/splits/DAIC_WOZ/val/audio"
val_text_dir = "/kaggle/input/splits-updated-223/splits/DAIC_WOZ/val/text"
test_audio_dir = "/kaggle/input/splits-updated-223/splits/DAIC_WOZ/test/audio"
test_text_dir = "/kaggle/input/splits-updated-223/splits/DAIC_WOZ/test/text"

In [10]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h", num_labels=7)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_dataset = DAIC_WOZ_Modal_Dataset(train_audio_dir, train_text_dir, processor, tokenizer)
val_dataset = DAIC_WOZ_Modal_Dataset(val_audio_dir, val_text_dir, processor, tokenizer)
test_dataset = DAIC_WOZ_Modal_Dataset(test_audio_dir, test_text_dir, processor, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

...

原始样本数: 8701, 有效样本数: 8701


过滤无效数据: 100%|██████████| 1072/1072 [00:01<00:00, 909.46it/s]


原始样本数: 1072, 有效样本数: 1072


过滤无效数据: 100%|██████████| 1058/1058 [00:01<00:00, 708.42it/s]

原始样本数: 1058, 有效样本数: 1058


In [11]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 加载预训练的 BERT 模型权重
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=7)

# 替换分类器层（假设你已经进行了微调）
bert_model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(0.5),
    torch.nn.Linear(bert_model.config.hidden_size, bert_model.config.num_labels)
)

# 加载你的微调权重（替换为你的路径）
bert_ckpt_path = "/kaggle/working/model/bert_meld_finetune_model.pth"
bert_model.load_state_dict(torch.load(bert_ckpt_path, map_location=torch.device('cpu')))

# load Wav2Vec2
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2Processor, AutoConfig
model_name = "facebook/wav2vec2-base-960h"
processor = Wav2Vec2Processor.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)
config.num_labels = 7
wav2vec2_model = Wav2Vec2ForSequenceClassification.from_pretrained(model_name, config=config)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-11-1ed4ef1bb246>:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We rec

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight', 'wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
from transformers import BertModel, Wav2Vec2Model
class MultimodalClassifier(nn.Module):
    def __init__(self, bert_model, wav2vec2_model):
        super(MultimodalClassifier, self).__init__()
        # 使用预训练的 BERT 模型
        self.bert = bert_model
        # 使用预训练的 Wav2Vec2 模型
        self.wav2vec2 = wav2vec2_model

        # 分类头
        self.classifier = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size + self.wav2vec2.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.7),
            nn.Linear(256, 7)  # 假设是7分类任务
        )
    
    def forward(self, text_input, audio_input):
        # 获取BERT的文本特征
        text_outputs = self.bert(**text_input, output_hidden_states=True)
        text_features = text_outputs.hidden_states[-1][:, 0, :]  # 获取[CLS] token的表示
        
        # 获取Wav2Vec2的音频特征
        audio_outputs = self.wav2vec2(audio_input, output_hidden_states=True)
        audio_features = audio_outputs.hidden_states[-1][:, 0, :]

        # 将音频和文本特征拼接
        combined_features = torch.cat((text_features, audio_features), dim=-1)

        # 分类
        logits = self.classifier(combined_features)
        return logits

In [13]:
from transformers import Trainer, TrainingArguments
from dataclasses import dataclass
import numpy as np
os.environ["WANDB_DISABLED"] = "true"  

@dataclass
class MultimodalCollator:
    def __call__(self, batch):
        return {
            
            "text_input_ids": torch.stack([torch.as_tensor(x["text_input_ids"]) for x in batch]),
            "text_attention_mask": torch.stack([torch.as_tensor(x["text_attention_mask"]) for x in batch]),
            "audio_input_values": torch.stack([torch.as_tensor(x["audio_input_values"]) for x in batch]),
            "labels": torch.tensor([x["labels"] for x in batch])
        }


class MultimodalClassifierAdapter(MultimodalClassifier):
    def forward(self, 
               text_input_ids=None, 
               text_attention_mask=None,
               audio_input_values=None,
               labels=None):
        # 将输入适配为原模型需要的格式
        text_input = {
            "input_ids": text_input_ids,
            "attention_mask": text_attention_mask
        }
        logits = super().forward(text_input, audio_input_values)
        
        loss = None
        if labels is not None:
            loss = torch.nn.functional.cross_entropy(logits, labels)
            
        return {"loss": loss, "logits": logits}





In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

model_path = "/kaggle/working/model/multimodal_MELD.pth" #model path here

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = MultimodalClassifierAdapter(bert_model, wav2vec2_model).to(device)
model.load_state_dict(torch.load(model_path))

# 定义优化器
optimizer = torch.optim.AdamW([
    {"params": model.bert.parameters(), "lr": 1e-6},
    {"params": model.wav2vec2.parameters(), "lr": 1e-6},
    {"params": model.classifier.parameters(), "lr": 1e-5}
], weight_decay=0.1)

# 配置训练参数
training_args = TrainingArguments(
    output_dir="/kaggle/working/finetuned_results",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="/kaggle/working/logs",
    fp16=True,
    learning_rate=1e-5,
    lr_scheduler_type='cosine',
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    logging_steps=200,
    logging_strategy="epoch",
    eval_steps=200,
    load_best_model_at_end=True,
)

# 自定义Trainer
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

    def evaluate(self, *args, **kwargs):
        output = super().evaluate(*args, **kwargs)
        self.lr_scheduler.step(output["eval_loss"])
        return output

# 定义评估指标计算函数
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {"accuracy": (preds == labels).mean()}

# 初始化Trainer
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=MultimodalCollator(),
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),
)

# 添加Early Stopping回调
class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience=3):
        self.patience = patience
        self.best_metric = None
        self.wait = 0

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics["eval_accuracy"]
        if self.best_metric is None or current_metric > self.best_metric:
            self.best_metric = current_metric
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                control.should_training_stop = True

trainer.add_callback(EarlyStoppingCallback(patience=3))

# 开始finetune
trainer.train()

<ipython-input-14-ad1c31bf361d>:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


Epoch,Training Loss,Validation Loss,Accuracy
1,1.427000,1.205447,0.559701
2,1.185700,1.052843,0.635261
3,1.041800,0.995369,0.659515


Error processing 370_92.wav: index 0 is out of bounds for axis 0 with size 0
Error processing 409_7.wav: index 0 is out of bounds for axis 0 with size 0
Error processing 339_38.wav: index 0 is out of bounds for axis 0 with size 0
Error processing 339_38.wav: index 0 is out of bounds for axis 0 with size 0
Error processing 409_7.wav: index 0 is out of bounds for axis 0 with size 0
Error processing 370_92.wav: index 0 is out of bounds for axis 0 with size 0
Error processing 339_38.wav: index 0 is out of bounds for axis 0 with size 0
Error processing 409_7.wav: index 0 is out of bounds for axis 0 with size 0
Error processing 370_92.wav: index 0 is out of bounds for axis 0 with size 0
Error processing 370_92.wav: index 0 is out of bounds for axis 0 with size 0
Error processing 339_38.wav: index 0 is out of bounds for axis 0 with size 0
Error processing 409_7.wav: index 0 is out of bounds for axis 0 with size 0
